[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Representation_Learning.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Representation Learning

Learning without labels: autoencoders that discover a signal's true coordinates, VAEs that turn compression into generation, and contrastive learning — the idea behind modern self-supervised pretraining. All demonstrated on a dataset whose true hidden structure we control, so we can *check* what was learned.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb), [Intro to ANN](./Intro_ANN/Intro_ANN.ipynb).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 (PCA — the linear ancestor).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# Dataset with KNOWN hidden structure: damped tones with 2 latent factors
# (frequency, decay) → 64-sample waveforms. Can 2 learned dimensions recover them?
def make_signals(n):
    freq = rng.uniform(0.05, 0.45, n)         # latent 1
    decay = rng.uniform(0.5, 4.0, n)          # latent 2
    t = np.arange(64)
    X = np.exp(-decay[:, None] * t / 64) * np.sin(2*np.pi*freq[:, None]*t)
    X += 0.02 * rng.standard_normal(X.shape)
    return X.astype(np.float32), freq, decay

X_np, freq, decay = make_signals(4000)
X = torch.from_numpy(X_np)
plt.figure(figsize=(8, 2))
for i in range(4): plt.plot(X_np[i], alpha=0.8)
plt.title("samples: damped tones, secretly governed by just 2 numbers")
plt.tight_layout(); plt.show()

**What just happened.** Four damped tones, each 64 samples long — and every one of them was generated from exactly **two numbers**: a frequency in $[0.05, 0.45]$ and a decay rate in $[0.5, 4.0]$, plus a little noise.

**That gap between 64 and 2 is the whole premise of the workshop.** The data *lives* in $\mathbb{R}^{64}$ but *occupies* a 2-dimensional surface inside it — a curved sheet parameterised by frequency and decay. Every real dataset has this property to some degree: images of faces are millions of pixels governed by far fewer factors of pose, lighting, and identity. **Representation learning is the business of finding the small number, without being told what it is.**

**Note the methodological luxury here, because it is deliberate and rare.** We *wrote* the generative process, so `freq` and `decay` are sitting in variables. When a network compresses to two dimensions later in this notebook, we can colour its latent space by the true factors and see whether it found them. Most representation-learning demos show a pretty scatter plot and assert that it means something; **this one can check**.

**Two design details in `make_signals` are worth catching.** The noise is $0.02\sigma$, so the **achievable reconstruction MSE is $0.0004$** — that number is the yardstick for every autoencoder in this notebook, and without it "MSE 0.01" is uninterpretable. And the two latent factors are drawn *independently* and uniformly, so the true latent distribution is a filled rectangle. Whether a learned latent space looks like one is a fair question to ask of the results.

**Look at the waveforms and predict what a linear method would do.** PCA would find the directions of greatest variance in $\mathbb{R}^{64}$ — but the map from (frequency, decay) to waveform is strongly nonlinear, so the 2-D manifold is *curved* and no 2-D linear subspace contains it. PCA would need many components to reach the same reconstruction error. **The nonlinearity in an autoencoder exists precisely to flatten curved sheets**, which is the one thing PCA structurally cannot do.

**One thing the plot quietly shows.** Signals with similar frequency but different decay look quite different, and signals with different frequency can overlap heavily early in the window. So the two factors are not separable by any simple summary statistic — you cannot just measure the zero-crossing rate and the envelope slope and call it done. The structure is real and the discovery problem is not trivial.

---
### 🕐 Session 1 of 3 — *Autoencoders* (~35 min)
**Goal:** compress through a bottleneck; check the latent space against the true factors.
**Builds on:** [ANN](./Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (VAEs).

---

## 2. The Bottleneck Game

💡 **Intuition.** An autoencoder learns two maps — encode (64 numbers → 2) and decode (2 → 64) — trained only to reconstruct its input. The bottleneck is the whole trick: to squeeze 64 samples through 2 numbers and back, the network is *forced* to discover the data's true coordinates. It's [PCA](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) with nonlinearity — and unlike PCA it can flatten curved manifolds.

In [ ]:

# YOUR CODE HERE


**What just happened.** Sixty-four numbers squeezed through a **2-dimensional** channel and back, with a final reconstruction MSE of **0.01048** against a signal variance of 0.133 — about 92% of the variance recovered from two numbers per waveform.

**But 92% is the flattering framing, and the honest yardstick is different.** The data was generated with additive noise of standard deviation 0.02, so the **best achievable MSE is $0.02^2 = 0.0004$**. We are at 0.0105 — **26× above the floor**. Since the true latent dimension is exactly 2, a perfect encoder–decoder pair would reach that floor. So the correct reading is: the bottleneck is the right size, and **the network has not converged**, not "the compression is nearly lossless".

**That distinction matters because it points at a different fix.** If the bottleneck were too small, no amount of training would help and the answer would be more latent dimensions. Here the answer is more epochs, a larger decoder, or a better learning-rate schedule — 60 passes over 4,000 points is not much. **Knowing which failure you have is the difference between a useful next experiment and a wasted one**, and the only way to know is to have the noise floor in hand.

**Note what the loss never mentions.** There is no reference to frequency, decay, sinusoids, or exponentials anywhere in the objective — just "reproduce your input". **The bottleneck is the entire teaching signal.** Sixty-four numbers cannot pass through a 2-D channel unless the network discovers the two numbers that actually determine the waveform, so the architecture, not the loss, does the work.

**Which is worth contrasting with PCA explicitly.** A *linear* autoencoder trained with squared error recovers exactly the PCA subspace — same answer, more expensive algorithm. The ReLUs here let the encoder flatten a **curved** manifold, and this dataset's manifold is strongly curved: the map from (frequency, decay) to waveform is trigonometric in one factor and exponential in the other. Two principal components would do far worse than 0.0105. **The nonlinearity is not decoration; it is the reason two dimensions suffice.**

**One caveat about the printed number itself.** `loss` is the value from the *last minibatch* of the last epoch, not an average over the dataset — so it carries batch-to-batch noise and is a slightly optimistic sample. Evaluating `((ae(X) - X)**2).mean()` over all 4,000 signals is one line and gives the number you would actually report.

In [ ]:
# The check most tutorials skip: does the latent space recover the TRUE factors?

# YOUR CODE HERE


**What just happened.** The 2-D latent space, coloured twice — once by the true frequency and once by the true decay. Both show **smooth colour gradients** across the cloud rather than a random speckle. The network found a coordinate system in which the generative factors vary continuously, and it did so having never been told those factors exist.

**This is the audit most representation-learning demos skip, and it is the reason the dataset was constructed with known latents.** A pretty scatter plot proves nothing — any encoder produces one. The question is whether the arrangement *means* anything, and the only way to answer it is to have ground truth on hand and colour by it. **Smooth gradient = structure recovered; speckle = the layout is arbitrary.**

**Now be precise about what was recovered, because the printed conclusion oversells slightly.** The latent axes are **not** frequency and decay. They are a warped, rotated, and probably entangled mixture of them — you can see this in the plots: the colour gradients are not aligned with $z_1$ and $z_2$, and they are not perpendicular to each other. The network recovered the **manifold**, not the **coordinates**.

**And that is not a training failure; it is a property of the objective.** Compose any invertible map $g$ with the encoder and $g^{-1}$ with the decoder and reconstruction is *exactly* unchanged. So the loss cannot distinguish between the true factorisation and any smooth reparameterisation of it — every one of them is an equally good optimum. **Nothing in a reconstruction loss prefers axis-aligned, independent factors**, so nothing produces them.

**Making it produce them is a named open problem.** *Disentanglement* — getting one latent axis per generative factor — needs extra structure: a stronger KL penalty ($\beta$-VAE), factorisation penalties (FactorVAE), architectural constraints, or weak supervision. And there is a well-known impossibility result: purely unsupervised disentanglement is not identifiable without inductive bias, because the reparameterisation freedom above is real. **Expecting the axes to be meaningful is expecting something the method never promised.**

**One experiment worth ninety seconds.** Re-run the whole training with a different seed and re-plot. The cloud will have a different shape, a different orientation, possibly a mirror flip — and the colour gradients will still be smooth. **The manifold is reproducible; the coordinates are not.** That single comparison teaches the identifiability point better than any amount of explanation.

**Finally, the question this sets up for Session 2.** Pick a point in the empty region between clusters, decode it, and look at the waveform. Nothing in the training objective ever required that point to decode to anything sensible — the latent space has **holes**. Filling them is precisely what the VAE's KL term is for, and it is what converts a compressor into a generator.

---
### 🕐 Session 2 of 3 — *Variational Autoencoders* (~40 min)
**Goal:** make the latent space a smooth, sampleable distribution; generate new signals.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (contrastive learning).

---

## 3. From Compression to Generation

💡 **Intuition.** A plain AE's latent space has holes — decode a random point and you may get garbage, because nothing forced the space to be *filled*. The VAE fixes this by encoding each input as a **distribution** (mean + spread) and penalizing the ensemble toward a standard Gaussian (a KL term — [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb)'s price of wrong beliefs, used as glue). Result: overlapping, hole-free latents — sample $z \sim \mathcal{N}(0, I)$, decode, and you *generate*. The reparameterization trick ($z = \mu + \sigma \epsilon$) keeps sampling differentiable.

In [ ]:

# YOUR CODE HERE


**What just happened.** Reconstruction **0.01077** — essentially identical to the plain autoencoder's 0.01048 — and **KL = 4.300**.

**Those two numbers together tell you this VAE is barely a VAE.** With $\beta = 0.001$, the KL contributes $0.001 \times 4.3 = 0.0043$ to a total loss of about 0.015. The regulariser is worth roughly a quarter of the reconstruction term, so the model is under almost no pressure to match the prior — and the matching reconstruction number confirms it: **behaviourally this is the Session 1 autoencoder with noise injected**.

**Read KL = 4.30 as a diagnostic, because it is one.** The KL measures how far the encoder's aggregate posterior sits from $\mathcal{N}(0, I)$. For a well-matched 2-D latent it should be **order 1**; 4.3 means the latent cloud is substantially wider and differently shaped than the prior. So the latent space this model actually uses is **not** the one the next cell samples from.

**That matters directly for the generation demo below.** Decoding a grid over $[-2,2]^2$ draws $z$ from the *prior*, but the encoder populated a different region. The decoder is smooth, so it will produce plausible-looking waveforms anyway — **plausible because of decoder smoothness, not because the prior is correct.** A stricter test is to encode real signals, look at where $\mu$ actually lands, and compare that cloud against a standard Gaussian. It will not match well at $\beta = 0.001$.

**So say what $\beta$ actually controls, since it is the one knob here.** Small $\beta$: sharp reconstructions, latent that ignores the prior, unreliable sampling. Large $\beta$: latent matches the prior and sampling works, at the cost of blurrier reconstructions. Push it too far and you hit **posterior collapse** — the encoder stops using the input, KL falls to zero, and the decoder emits the dataset mean for every $z$. **Both ends are real failure modes, and the usable range is narrow.** The $\beta$-VAE literature is precisely the study of this dial.

**The reparameterisation line is the piece of engineering worth pausing on.** `z = mu + torch.exp(0.5*logv) * torch.randn_like(mu)` exists because you cannot backpropagate through "draw a sample" — sampling is not differentiable in the distribution's parameters. Writing $z = \mu + \sigma\varepsilon$ pushes all the randomness into a **parameter-free** term, leaving a clean gradient path to $\mu$ and $\sigma$. One line of algebra, without which VAEs are untrainable.

**The experiment that turns this cell into a measurement takes ninety seconds.** Re-run with $\beta = 1.0$ and compare: KL should fall toward 1, reconstruction should worsen noticeably, and the decoded grid in the next cell should become *more* coherent rather than less. **Two runs and a two-row table make the trade-off concrete** in a way no amount of description does.

In [ ]:
# Generate: walk a grid of the latent space and decode — brand-new signals

# YOUR CODE HERE


**What just happened.** Sixteen latent points on a $4\times4$ grid over $[-2,2]^2$, decoded into sixteen waveforms — **none of which is in the training set**. Move across a row and the frequency shifts; move down a column and the decay changes. The latent space is **smooth**: nearby $z$ give nearby signals, and the morphing is continuous rather than jumping between memorised examples.

**That smoothness is the whole achievement, and it is what a plain autoencoder does not guarantee.** Session 1's model was free to scatter its training data anywhere, holes included — decode an unvisited point and you may get nothing. Here the encoder outputs *distributions* with overlapping support, so neighbouring regions are forced to decode to similar things. **Compression became generation because the latent space was made continuous.**

**Now the caveat, which follows directly from the KL of 4.30 in the previous cell.** This grid samples from the **prior** $\mathcal{N}(0,I)$, but at $\beta = 0.001$ the encoder's aggregate posterior is much wider than that — KL 4.3 in two dimensions is far from the order-1 value a matched latent would give. So the $z$ values being decoded here are **not drawn from the distribution the encoder actually produces**. The waveforms look plausible because the decoder is a smooth function, not because prior and posterior agree.

**Which means "sampled from the model" is a stronger claim than this cell establishes.** The honest version: *these are decodings of a systematic sweep through latent space*, which demonstrates smoothness and interpolation. Genuine sampling would require the prior to match the aggregate posterior, and the KL says it does not.

**The check that would settle it is two lines.** Encode the real data, collect the $\mu$ values, and scatter them with the grid overlaid. If the grid lands inside the occupied cloud, the sampling claim holds; if the cloud is much broader or offset, you are decoding regions the model never learned about. **Do the check rather than trusting the pictures** — this is the same discipline as the latent audit in Session 1, applied to generation instead of structure.

**Look for the visible signature of the VAE's known weakness too.** Gaussian likelihoods average over plausible outputs, so VAE samples tend toward the **smooth and slightly bland** — here that shows up as waveforms with clean envelopes and no high-frequency texture, and in image VAEs as the characteristic blur. That failure is exactly what [diffusion models](./Diffusion_Models.ipynb) fix, and having seen the VAE first is what makes the diffusion objective legible as a response to it.

**Finally, note how much of modern generative modelling is already visible here.** A prior you can sample, a decoder, and a training objective that keeps the two compatible. Latent diffusion — Stable Diffusion included — runs the diffusion process inside a VAE's latent space, so this architecture did not get replaced so much as demoted to a component.

---
### 🕐 Session 3 of 3 — *Contrastive Learning* (~40 min)
**Goal:** learn representations by agreement between augmented views — no reconstruction at all.
**Builds on:** Session 2.

---

## 4. Learning by Comparison

💡 **Intuition.** Reconstruction wastes capacity on details you don't care about (exact noise, phase). Contrastive learning changes the question: make two *augmented views* of the same signal map **close**, and views of different signals map **apart**. What survives is exactly what your augmentations declare irrelevant — invariance is a *design choice*. This is the engine of SimCLR/CLIP-style pretraining: no labels, just the physics of 'what shouldn't matter'.

In [ ]:

# YOUR CODE HERE


**What just happened.** Final InfoNCE loss **3.404** against a chance level of $\ln(256) = 5.55$ and a perfect score of 0. The model got meaningfully better than random at matching a signal to its own augmented twin among 255 distractors — and it is **nowhere near solving the task**.

**Read the loss as what it literally is: a 256-way classification.** Each row of `logits` scores one view against all 256 candidates; the correct answer is the diagonal. So 3.404 corresponds to assigning roughly $e^{-3.4} \approx 3\%$ probability to the right partner — about 8× better than the 0.4% chance rate, and a long way from confident identification.

**The reason it plateaus there is structural, not a training failure, and it is worth working out rather than accepting.** The encoder outputs 2-D vectors that are then **normalised to unit length**. A unit vector in $\mathbb{R}^2$ lives on the **circle** — which is **one-dimensional**. One number: the angle. But the data has **two** generative factors. **The embedding is mathematically incapable of representing both**, so it keeps the one the task rewards most and discards the other. That is a capacity argument and it predicts exactly the floor observed.

**Which makes the fix a one-character edit and the best experiment in the notebook.** Change `nn.Linear(32, 2)` to `nn.Linear(32, 3)`: the embedding becomes a **sphere**, genuinely 2-dimensional, with room for both factors. The loss should fall noticeably. **Ninety seconds of compute converts a stated limitation into a measured one**, and confirms that the bottleneck was geometry rather than optimisation.

**Now the sentence that is the real content of the session.** *What survives is exactly what your augmentations declare irrelevant.* `augment` rolls the signal in time, rescales the gain, and adds noise — declaring all three to be nuisance. Frequency and decay survive because nothing touched them. **Invariance is a design choice written into the augmentation function**, and it is the most consequential decision in self-supervised learning: destroy your signal of interest with a careless augmentation and you learn nothing, choose well and you get CLIP.

**Note also why the batch size is not incidental.** The 255 non-matching items in each batch *are* the negatives. Larger batches mean harder discrimination and better representations, which is why SimCLR-scale methods use batches in the thousands and why memory banks and momentum encoders exist — they are all ways to obtain more negatives than fit in memory at once.

**One difference from real implementations worth knowing before reading the papers.** SimCLR inserts a **projection head** between the encoder and the contrastive loss, then discards it and uses the pre-projection features downstream. The reason is that the contrastive objective throws away information a downstream task may want, and the head absorbs that loss. Omitted here for clarity, and standard everywhere in practice.

In [ ]:

# YOUR CODE HERE


**What just happened.** The embedding is a **ring** — every point sits at radius 1, because the encoder normalises its output — and the colour sweeps smoothly around it with the true frequency. Signals of similar frequency landed at similar angles, and no label was ever used.

**Two things earned that arrangement, and they are worth separating.** The augmentations declared shift, gain, and noise irrelevant, so those were discarded. And the InfoNCE loss required each signal's two views to land near each other while pushing different signals apart, so what remains must distinguish signals. **Frequency organises the circle because it is what survived the augmentations and what the task needed** — not because anyone mentioned frequency.

**Now look at what is *not* here, since the missing thing is the lesson.** Colour the same plot by `decay` and the gradient will be far weaker or absent. The reason is the geometry: unit vectors in $\mathbb{R}^2$ live on a circle, which is **one-dimensional** — a single angle. The data has **two** generative factors, and one coordinate cannot carry two. **The embedding did not choose to ignore decay; it had nowhere to put it.**

**That also explains the loss floor from the previous cell.** InfoNCE stalled at 3.40 against a chance of 5.55, and no amount of further training moves it, because the representation is at capacity. One-character fix: output 3 dimensions instead of 2, turning the circle into a sphere with genuine room for both factors. **A capacity argument that predicts a number, and a one-line experiment that tests it** — worth running before accepting either claim.

**Compare this embedding against the autoencoder's from Session 1, because the contrast is the point of the workshop.** The autoencoder had to preserve **everything** needed to reconstruct the waveform — frequency, decay, and a good deal of the noise and phase. This encoder preserves only what the augmentations left intact. Same data, same architecture family, completely different representations, **and the difference is entirely in the objective**.

**Which is why augmentation design is the real work in self-supervised learning.** Had `augment` also stretched the time axis, frequency would have become a nuisance variable and this plot would organise by something else — or by nothing. **You get to specify what the representation ignores, and that specification is the whole model.** SimCLR's augmentation ablations are the most-cited table in the paper for exactly this reason, and CLIP's "augmentation" is the pairing of an image with its caption: two views of one thing, declaring everything not shared between them irrelevant.

**A last note on reading the picture honestly.** Smooth colour around the ring shows that frequency is *encoded*, not that it is encoded *well* — the mapping could be badly warped, compressing part of the range into a small arc. A stronger check is to fit a simple regressor from the embedding angle to the true frequency and report its $R^2$. That takes two lines and turns "the colours look organised" into a number.

## 5. Conclusion

Bottlenecks force discovery; the VAE's KL glue turns discovery into generation; contrastive losses let *you* choose what representation ignores. Three label-free recipes — the third is how frontier models pretrain.

---
## Where next

- [Diffusion Models](./Diffusion_Models.ipynb) — generation taken to the modern state of the art.
- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — next-token prediction as yet another self-supervised objective.